In [1]:
using StatsBase
using JLD
using DataFrames
using Statistics

import IterTools

include("./model.jl")

get_sub_data (generic function with 1 method)

In [2]:

function get_block(true_planet)
    block =[]
    for p in true_planet
        if p < 20
            append!(block,1)
        elseif p < 40
            append!(block,2)
        elseif p < 60
            append!(block,3)
        elseif p < 80
            append!(block,4)
        elseif p < 100
            append!(block,5)
        end
    end
    return block
end


function chunk_galaries(galaxies)
    groups=[]
    for i in IterTools.groupby(x -> x, galaxies)
        push!(groups,i)
    end
        return groups
end

function number_planet_in_galaxy(galaxies)
    chunks = chunk_galaries(galaxies)
    return [i for galaxy in chunks for i in 1:length(galaxy)]
end

number_planet_in_galaxy (generic function with 1 method)

In [3]:
function three_patch_inference(alpha)
    if alpha
        params = [1,5,2]
        filename = "model_results/show_inference_1.csv"
    else
        params = [0,5,2]
        filename = "model_results/show_inference_0.csv"
    end
    
    data = get_sub_data(76)
    num_particles = 1
    b,x,y,z=crp_adaptiveDiscount(data,params,num_particles)
    opt_prt = optimal_policy(b)
    diff = b.prt - opt_prt
    
    # put in dataframe and save
    df = DataFrame(Dict("true_planet"=> b.true_planet,"galaxy"=> b.galaxy,"prt"=>b.prt,
        "opt_prt"=>opt_prt, "diff" => diff))

    gdf = groupby(df, :galaxy)
    prt_avg = combine(gdf, :diff => mean)


    df = DataFrame(Dict("true_planet"=> b.true_planet,"galaxy"=> b.galaxy,"prt"=>b.prt,
        "opt_prt"=>opt_prt, "diff" => diff,"pred_decay"=>x,"uncertainty"=>y,"true_decay"=>z))
    
    CSV.write(filename,df)
    return  
end

three_patch_inference (generic function with 1 method)

In [4]:
function sub_params(model, sub_num, output, fit_params, file_path)
    sub_data = get_sub_data(sub_num)
    if cmp(model,"adaptive_discount") == 0
        if fit_params # load in the best fitting parameters 
            # println(sub_num)
            d=load(string("fit_params/",file_path,"/sub",string(sub_num),".jld"))
            params = d["res"]
            # println(params)
        else 
            params = [5,1,0.6]
        end
        num_particles = 1
        # println("sub_num"=> sub_num,"alpha"=> params[1],"gamma_base"=> params[2],"gamma_coef"=> params[3])
        b = crp_adaptiveDiscount(sub_data,params,num_particles);
        df_params = DataFrame(Dict("sub_num"=> sub_num,"alpha"=> params[1],"gamma_base"=> params[2],"gamma_coef"=> params[3]))
        # println(b)
    elseif cmp(model,"mvt") == 0
        if fit_params
            d=load(string("fit_params/MVT_learn_1_20_3/sub",string(sub_num),".jld"))
            params = d["res"]
        else
            params = [0.1,10,1]           
        end
        b = MVT_learn(sub_data,params);
        df_params = DataFrame(Dict("sub_num"=> sub_num,"alpha"=> params[1],"beta"=> params[2],"c"=> params[3]))

    elseif cmp(model,"td") == 0
        if fit_params
            d=load(string("fit_params/td_1_20_1_3/sub",string(sub_num),".jld"))
            params = d["res"]
        else
            params = [0.5,10,0.97,2]
        end
        b = TD(sub_data,params);
        df_params = DataFrame(Dict("sub_num"=> sub_num,"alpha"=> params[1],"beta"=> params[2],"gamma"=> params[3],"c"=> params[4]))

    end

    opt_prt = optimal_policy(b);
    diff = b.prt - opt_prt;
                
    df = DataFrame(Dict("true_planet"=> b.true_planet,"galaxy"=> b.galaxy,"block"=> get_block(b.true_planet),"prt"=>b.prt,
        "opt_prt"=>opt_prt, "diff" => diff))
    df[!,"n_galaxy"] = number_planet_in_galaxy(df.galaxy)
       
    
    if cmp(output,"avg_and_params") == 0
        gdf = groupby(df, :galaxy)
        prt_avg = combine(gdf, :diff => mean)
        insertcols!(prt_avg,       # DataFrame to be changed
        1,                # insert as column 1
        :sub_num => ones(size(prt_avg)[1])*sub_num,   # populate as "Day" with 1,2,3,..
        makeunique=true) 
    
        insertcols!(df_params,       # DataFrame to be changed
        1,                # insert as column 1
        :sub_num => ones(size(df_params)[1])*sub_num,   # populate as "Day" with 1,2,3,..
        makeunique=true) 

        return prt_avg, df_params
    elseif cmp(output,"full") == 0
        insertcols!(df,       # DataFrame to be changed
        1,                # insert as column 1
        :sub_num => ones(size(df)[1])*sub_num,   # populate as "Day" with 1,2,3,..
        makeunique=true) 

        return df 
    end
end

function all_subs(model,subs,output,fit_params,file_path)
    if cmp(output,"avg_and_params") == 0
        df_prt = DataFrame()
        df_params = DataFrame()
        for sub in subs
            try
                # println("try")
                prt, params = sub_params(model,sub,output,fit_params,file_path)
                df_prt = vcat(df_prt,prt)
                # print(df_prt)
                df_params = vcat(df_params,params)
            catch err
            end
        end
        return df_prt, df_params
    elseif cmp(output,"full") == 0
        df_prt = DataFrame()
        for sub in subs
            for n_sim=1:50
                try
                    prt = sub_params(model,sub,output,fit_params,file_path)
                    insertcols!(prt,       # DataFrame to be changed
                        1,                # insert as column 1
                    :n_sim => ones(size(prt)[1])*n_sim,   # populate as "Day" with 1,2,3,..
                    makeunique=true) 
                    df_prt = vcat(df_prt,prt)
                catch err
                end
            end
        end
        return df_prt
    end
end 

all_subs (generic function with 1 method)

# REMEMBER to change the ***csv*** name in the function ***get_sub_data*** on ***model.jl*** and ***restart*** the notebook, especially if you are fitting one dataset but grabbing another dataset in this file!!

# online data

In [10]:
# sub 81 is removed due to missing 3rd?
# sub 155 is removed due to missing 2nd/3rd block?

subs=[  1,   2,   3,   5,   7,  10,  11,  12,  13,  14,  16,  18,  20,
    21,  23,  27,  28,  29,  32,  33,  35,  36,  37,  38,  39,  40,
    41,  42,  44,  45,  46,  47,  48,  50,  51,  53,  54,  56,  57,
    58,  60,  61,  62,  64,  65,  66,  67,  68,  69,  70,  71,  72,
    73,  74,  75,  76,  77,  78,  82,  83,  85,  86,  87,  89,  91,
    92,  93,  94,  95,  96,  97,  98,  99, 100, 101, 102, 104, 105,
    108, 110, 111, 112, 113, 114, 117, 118, 119, 120, 121, 122, 123,
    125, 127, 129, 130, 131, 136, 137, 138, 139, 141, 142, 143, 144,
    145, 146, 147, 148, 149, 150, 151, 152, 156, 158, 161, 163, 164,
    167, 170, 171, 172, 173, 174, 176, 177, 178, 180, 181, 182, 184,
    185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 197, 198,
    200, 201, 202, 203, 204, 205, 206, 207, 209, 212, 213, 214, 217,
    219, 222, 223, 225, 226, 227, 229, 232, 233, 234, 235, 236, 237,
    238, 241, 243, 244, 245, 248, 249, 250, 251, 252, 254, 257, 258,
    259, 260, 261, 262, 263, 268, 269, 273, 274, 275, 279, 280, 281,
    282, 284, 285, 286, 289, 293, 296, 298, 300, 301, 303, 304, 305,
    308, 309, 310, 311, 312, 314, 316, 317, 319, 320, 321, 323, 324,
    325, 327, 328, 330, 331, 332, 336, 338, 339, 340, 341, 342, 345,
    346, 347, 350, 352, 355, 358, 362, 363, 365, 366, 367, 368, 369,
    371, 372, 373, 374, 377, 379, 381, 386, 387, 388, 392, 394, 396,
    399, 400, 401, 403, 409, 411, 413, 415, 421, 423, 424, 427, 431,
    439, 445, 447, 450, 451, 460, 463, 464, 466, 467, 468, 473, 488,
    490, 492, 497, 498, 505, 506, 508, 509, 510, 511, 513]

prt,params = all_subs("adaptive_discount",subs,"avg_and_params",true,"adaptive_discount_online_08_21_24")
CSV.write("model_results/prt_val_adaptive_discount_online_08_21_24.csv",prt)
CSV.write("model_results/params_val_adaptive_discount_online_08_21_24.csv",params)

# prt,params = all_subs("mvt",subs,"avg_and_params",true)
# CSV.write("model_results/prt_val_MVT_learn.csv",prt)
# CSV.write("model_results/params_val_MVT_learn.csv",params)

# prt,params = all_subs("td",subs,"avg_and_params",true)
# CSV.write("model_results/prt_val_TD.csv",prt)
# CSV.write("model_results/params_val_TD.csv",params)

"model_results/params_val_adaptive_discount_online_08_21_24.csv"

In [11]:
params

Row,sub_num_1,alpha,gamma_base,gamma_coef,sub_num
,Float64,Float64,Float64,Float64,Int64
1,1.0,7.4707,-1.91797,-2.53711,1
2,2.0,6.39648,0.496094,-0.767578,2
3,3.0,9.96094,-1.95312,-1.14844,3
4,5.0,1.66016,-1.88281,-0.410156,5
5,7.0,2.1582,-1.54297,-0.0996094,7
6,10.0,0.0390625,9.95312,1.61719,10
7,11.0,0.166016,-1.40234,1.51758,11
8,12.0,5.57617,0.167969,-1.16602,12
9,13.0,0.292969,-0.804688,-0.0117188,13


# in-person data

In [9]:
function get_sub_data(sub_num)
    all_data=DataFrame(CSV.File("data/all_data_ip_07_02_24.csv",delim=',')) # YC
    sub_data = all_data[in(sub_num).(all_data.sub_num),:]
end

get_sub_data (generic function with 1 method)

In [10]:
subs=[0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 11, 12, 13, 14, 15, 16, 17,
18, 19, 20, 21, 22, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 37,
38, 39, 40, 41, 42, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 56, 57,
58, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 74]

prt,params = all_subs("adaptive_discount",subs,"avg_and_params",true,"adaptive_discount_ip_07_02_24")
CSV.write("model_results/prt_val_adaptive_discount_ip_07_02_24.csv",prt)
CSV.write("model_results/params_val_adaptive_discount_ip_07_02_24.csv",params)

"model_results/params_val_adaptive_discount_ip_07_02_24.csv"

In [11]:
params

Row,sub_num_1,alpha,gamma_base,gamma_coef,sub_num
,Float64,Float64,Float64,Float64,Int64
1,0.0,7.91016,8.61719,-1.16016,0
2,1.0,0.195312,0.390625,0.585938,1
3,2.0,0.195312,0.390625,0.585938,2
4,3.0,2.87109,0.601562,0.128906,3
5,4.0,7.27539,0.378906,0.251953,4
6,5.0,0.0390625,9.95312,1.61719,5
7,6.0,0.0390625,9.95312,1.61719,6
8,7.0,9.17969,0.859375,0.0703125,7
9,8.0,1.5625,-0.125,0.1875,8


In [12]:
prt

Row,sub_num,galaxy,diff_mean
,Float64,Any,Float64
1,0.0,0,0.0
2,0.0,1,-0.0769231
3,0.0,2,-0.4
4,1.0,0,0.333333
5,1.0,1,1.2
6,1.0,2,1.78947
7,2.0,0,0.307692
8,2.0,1,1.12
9,2.0,2,1.68421
